# F1 Comparison: Step-3-Only vs API Pipeline

Loads `metrics.csv` files from both reproduction runs and produces a side-by-side F1 comparison table.

| Source | Run |
|--------|-----|
| Wikipedia BM25, Semantic | Step-3-only (paper evidence) |
| PubMed BM25, Semantic | Step-3-only (paper evidence) |
| Google | Step-3-only (paper evidence) |
| Wikipedia API | API pipeline (independently retrieved) |
| PubMed API | API pipeline (independently retrieved) |

In [1]:
import os
import csv
import pandas as pd

try:
    import google.colab
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

if RUNNING_ON_COLAB:
    BASE_PATH = '/content/drive/MyDrive/NLP-Group-Project'
else:
    BASE_PATH = '../..'

STEP3_PATH  = os.path.join(BASE_PATH, 'reproduction', 'step3-only',   'results')
API_PATH    = os.path.join(BASE_PATH, 'reproduction', 'api-pipeline', 'results')
OUTPUT_PATH = os.path.join(BASE_PATH, 'reproduction', 'overview')

DATASETS = ['scifact', 'pubmedqa', 'healthfc', 'covert']
print(f'Step-3-only results: {os.path.abspath(STEP3_PATH)}')
print(f'API pipeline results: {os.path.abspath(API_PATH)}')

Step-3-only results: /content/drive/MyDrive/NLP-Group-Project/reproduction/step3-only/results
API pipeline results: /content/drive/MyDrive/NLP-Group-Project/reproduction/api-pipeline/results


In [2]:
def load_metrics(path):
    """Load a metrics.csv and return {dataset: f1} dict."""
    results = {}
    with open(path, 'r', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            results[row['dataset']] = float(row['f1'])
    return results

# Paper F1 values from Tables 3 & 4 of Vladika & Matthes (EACL 2024)
paper = {
    'Wiki BM25':     {'scifact': 74.8, 'pubmedqa': 73.1, 'healthfc': 73.1, 'covert': 75.2},
    'Wiki Semantic': {'scifact': 75.4, 'pubmedqa': 73.2, 'healthfc': 76.5, 'covert': 82.5},
    'PubMed BM25':   {'scifact': 76.1, 'pubmedqa': 70.3, 'healthfc': 69.7, 'covert': 79.5},
    'PubMed Sem':    {'scifact': 76.8, 'pubmedqa': 74.5, 'healthfc': 72.0, 'covert': 76.2},
    'Google':        {'scifact': 82.7, 'pubmedqa': 78.5, 'healthfc': 74.5, 'covert': 72.3},
}

ours = {
    'Wiki BM25':     load_metrics(os.path.join(STEP3_PATH, 'Wikipedia BM25',    'metrics.csv')),
    'Wiki Semantic': load_metrics(os.path.join(STEP3_PATH, 'Wikipedia Semantic', 'metrics.csv')),
    'PubMed BM25':   load_metrics(os.path.join(STEP3_PATH, 'PubMed BM25',       'metrics.csv')),
    'PubMed Sem':    load_metrics(os.path.join(STEP3_PATH, 'PubMed Semantic',    'metrics.csv')),
    'Google':        load_metrics(os.path.join(STEP3_PATH, 'Google',             'metrics.csv')),
    'Wikipedia API': load_metrics(os.path.join(API_PATH,   'Wikipedia API',      'metrics.csv')),
    'PubMed API':    load_metrics(os.path.join(API_PATH,   'PubMed API',         'metrics.csv')),
}

print('Loaded all sources.')

Loaded all sources.


In [3]:
# Build multi-level columns: (source, Paper/Ours)
# Sources with paper values get both columns; API sources get Ours only.
columns = []
data    = {}

for src in ['Wiki BM25', 'Wiki Semantic', 'PubMed BM25', 'PubMed Sem', 'Google']:
    columns += [(src, 'Paper'), (src, 'Ours')]
    for ds in DATASETS:
        data[(src, 'Paper')] = data.get((src, 'Paper'), []) + [paper[src][ds]]
        data[(src, 'Ours')]  = data.get((src, 'Ours'),  []) + [ours[src][ds]]

for src in ['Wikipedia API', 'PubMed API']:
    columns += [(src, 'Ours')]
    for ds in DATASETS:
        data[(src, 'Ours')] = data.get((src, 'Ours'), []) + [ours[src][ds]]

df = pd.DataFrame(data, index=DATASETS)
df.index.name = 'Dataset'
df.columns = pd.MultiIndex.from_tuples(columns)

# Add average row
df.loc['Average'] = df.mean()
df = df.round(1)

# Highlight best Ours value per dataset row across all Ours columns
ours_cols = [c for c in df.columns if c[1] == 'Ours']

def highlight_best_ours(row):
    styles = [''] * len(row)
    best = max(row[c] for c in ours_cols)
    for i, c in enumerate(row.index):
        if c[1] == 'Ours' and row[c] == best:
            styles[i] = 'font-weight: bold; background-color: #d4edda'
    return styles

styled = df.style.apply(highlight_best_ours, axis=1)
print('F1 scores (%) — green = best Ours result per dataset')
styled

F1 scores (%) — green = best Ours result per dataset


In [4]:
out_path = os.path.join(OUTPUT_PATH, 'f1_comparison.xlsx')
df.to_excel(out_path)
print(f'Saved: {out_path}')
print()
print(df.to_string())

Saved: /content/drive/MyDrive/NLP-Group-Project/reproduction/overview/f1_comparison.xlsx

         Wiki BM25       Wiki Semantic       PubMed BM25       PubMed Sem       Google       Wikipedia API PubMed API
             Paper  Ours         Paper  Ours       Paper  Ours      Paper  Ours  Paper  Ours          Ours       Ours
Dataset                                                                                                              
scifact       74.8  74.8          75.4  75.4        76.1  76.0       76.8  76.8   82.7  72.6          78.1       76.5
pubmedqa      73.1  70.4          73.2  72.1        70.3  68.4       74.5  70.8   78.5  58.7          72.5       71.8
healthfc      73.1  73.1          76.5  76.5        69.7  70.0       72.0  74.5   74.5  74.2          74.5       71.9
covert        75.2  80.4          82.5  82.5        79.5  79.0       76.2  77.8   72.3  54.0          68.0       70.2
Average       74.0  74.7          76.9  76.6        73.9  73.4       74.9  75.0   77